<a href="https://colab.research.google.com/github/ShreyaMathur19/Clustered-Diagonal-Segment-Format-CDSF-/blob/main/CDSF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**CDSF (Compressed Diagonal Storage with Fragments)**

CDSF stores only non-zero diagonal fragments (clusters) instead of full diagonals, reducing storage of unnecessary zeros. We converted the matrix from COO to CDSF, performed SpMV using a Numba-optimized kernel, and measured execution time and peak memory usage to analyze its efficiency compared to CDS.

In [ ]:
from scipy.io import mmread
import numpy as np
from scipy.sparse import coo_matrix
from time import perf_counter
import numba as nb

In [ ]:
# Load the matrix from an .mtx file
A = mmread("wang3.mtx")
A = coo_matrix(A)

In [ ]:
#COO to CDSF

import numpy as np
from scipy.sparse import coo_matrix

def encode_cdsf(A):
    """
    Encode a sparse matrix A into CDSF without a bandwidth parameter.
    Scans all diagonals; clusters are maximal runs of consecutive nonzeros
    along a diagonal (i.e., (i,k),(i+1,k+1),...).

    Returns:
        row_indices : int32  [num_clusters]  start row of each cluster
        col_indices : int32  [num_clusters]  start col of each cluster
        lengths     : int32  [num_clusters]  length (number of values) of each cluster
        values      : float64 [sum(lengths)] concatenated cluster values in order
    """
    # Ensure COO; coalesce duplicates; force float64 values
    A = A.tocoo(copy=False)
    if hasattr(A, "sum_duplicates"):
        A.sum_duplicates()
    row = A.row.astype(np.int32, copy=False)
    col = A.col.astype(np.int32, copy=False)
    data = A.data.astype(np.float64, copy=False)

    nnz = data.size
    if nnz == 0:
        # Empty matrix
        return (np.empty(0, dtype=np.int32),
                np.empty(0, dtype=np.int32),
                np.empty(0, dtype=np.int32),
                np.empty(0, dtype=np.float64))

    # Diagonal offset: col - row
    off = (col - row).astype(np.int32, copy=False)

    # Sort by (offset, row) so each diagonal is contiguous and increasing
    # np.lexsort uses last key as primary; we want offset primary, row secondary.
    order = np.lexsort((row, off))
    off_s  = off[order]
    row_s  = row[order]
    col_s  = col[order]
    data_s = data[order]

    # Detect starts of clusters:
    # A new cluster starts at the first element, or whenever the diagonal changes,
    # or when rows are not consecutive (break in the diagonal run).
    # is_new_cluster[k] = True iff k==0 or off_s[k]!=off_s[k-1] or row_s[k]!=row_s[k-1]+1
    is_new = np.empty(nnz, dtype=bool)
    is_new[0] = True
    same_diag = off_s[1:] == off_s[:-1]
    consec_row = row_s[1:] == (row_s[:-1] + 1)
    is_new[1:] = ~(same_diag & consec_row)

    # Cluster start indices and lengths (run-length encoding)
    starts = np.nonzero(is_new)[0]
    ends   = np.empty_like(starts)
    ends[:-1] = starts[1:]
    ends[-1]  = nnz
    lengths = (ends - starts).astype(np.int32, copy=False)

    # Cluster starting coordinates
    row_indices = row_s[starts].astype(np.int32, copy=False)
    col_indices = col_s[starts].astype(np.int32, copy=False)

    # Values are already in cluster-concatenated order
    values = data_s  # float64

    return row_indices, col_indices, lengths, values


In [ ]:
#CDSF SPMV
@nb.njit(cache=True, fastmath=True)
def cdsf_spmv_numba(row_idx, col_idx, lengths, values, nrows, ncols, x):
    y = np.zeros(nrows, dtype=np.float64)
    pos = 0
    K = lengths.shape[0]
    for k in range(K):
        r0 = row_idx[k]
        c0 = col_idx[k]
        L  = lengths[k]
        for i in range(L):
            y[r0 + i] += values[pos + i] * x[c0 + i]
        pos += L
    return y

In [ ]:
#SPMV time
def time_cdsf_spmv_numba(row_idx, col_idx, lengths, values, shape,reps=20, seed=42):
    """
    Times y = A @ x for CDSF (Numba kernel).
    Inputs:
      - row_idx, col_idx, lengths: int32 arrays
      - values: float64 array
      - shape: (nrows, ncols)
    Returns: ms per call (float)
    """
    nrows, ncols = shape
    rng = np.random.default_rng(seed)
    x = rng.standard_normal(ncols).astype(np.float64)

    # --- Warm-up #1: trigger JIT compilation (excluded from timing)
    _ = cdsf_spmv_numba(row_idx, col_idx, lengths, values, nrows, ncols, x)

    # --- Warm-up #2: one steady-state run (also excluded)
    _ = cdsf_spmv_numba(row_idx, col_idx, lengths, values, nrows, ncols, x)

    # --- Timed runs
    t0 = perf_counter()
    for _ in range(reps):
        y = cdsf_spmv_numba(row_idx, col_idx, lengths, values, nrows, ncols, x)
    t_ms = (perf_counter() - t0) / reps * 1000.0

    print(f"CDSF SpMV (Numba): {t_ms:.2f} ms per call  |  shape={shape}, clusters={len(lengths)}, values={values.size}")
    return t_ms

In [ ]:
#Time and peak memory for coo to cdsf
def time_and_peak_mem_coo_to_cdsf(A_coo, reps=5, interval=0.0001):
    """
    Measures average time (ms) and peak memory (KB) for COO->CDSF encoding.
    Uses memory_profiler to capture peak RSS delta.
    """
    times_ms = []
    peak_deltas_kb = []
    arrays = None

    # Warm-up
    _ = encode_cdsf(A_coo)

    for run in range(reps):
        t0 = perf_counter()
        mem_trace, arrays = memory_usage(
            (encode_cdsf, (A_coo,), {}), retval=True, interval=interval
        )
        elapsed_ms = (perf_counter() - t0) * 1000.0
        peak_delta_kb = (max(mem_trace) - mem_trace[0]) * 1024

        print(f"Run {run+1}: start={mem_trace[0]*1024:.2f} MB, peak={max(mem_trace)*1024:.2f} MB, delta={peak_delta_kb:.2f} KB, time={elapsed_ms:.2f} ms")

        times_ms.append(elapsed_ms)
        peak_deltas_kb.append(peak_delta_kb)

    avg_ms = float(np.mean(times_ms))
    peak_kb = float(max(peak_deltas_kb))

    print("\n=== Summary COO->CDSF ===")
    print(f"Average time: {avg_ms:.2f} ms per call (over {reps})")
    print(f"Peak memory during conversion: {peak_kb:.2f} KB")

    return arrays, avg_ms, peak_kb